# 🐱 Cat Detection — Selective Search + MobileNetV2 (R-CNN style)

Pipeline: **Selective Search** proposes candidate regions → a **MobileNetV2** classifier decides *cat / no_category* for each region → **Non-Maximum Suppression** keeps the best boxes.

**Before running**
1. Download the cat dataset (images `cat.N.jpg` + annotations `cat.N.xml`) and put it inside an `archive/` folder next to this notebook.
2. Put a photo of your own named `sample_input.jpg` next to this notebook (used for the final detection demo).
3. `pip install -r requirements.txt` (needs **opencv-contrib-python**, not plain opencv-python).

In [ ]:
import random
import re
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# ---------------- Configuration (edit here only) ----------------
DATA_DIR = Path("archive")         # folder with cat.N.jpg and cat.N.xml (sub-folders like "Asirra_ cat vs dogs" are found automatically)
TRAIN_DIR = Path("training")       # crops generated below (large, not committed to git)
SAMPLE_IMAGE = Path("sample_input.jpg")
MODEL_PATH = Path("cat_detector.keras")

IMG_SIZE = (224, 224)              # input size MobileNetV2 was pre-trained on
BATCH_SIZE = 32
EPOCHS = 10
MAX_IMAGES = 350                   # dataset images to process (None = all, very slow)
MAX_FILE_SIZE = 100_000            # only use images < 100 KB (faster region proposals)
REGENERATE_CROPS = False           # True = delete and rebuild the training crops
SEED = 42

keras.utils.set_random_seed(SEED)
random.seed(SEED)

## 1. Helper functions

In [ ]:
def extract_ground_truth(xml_path):
    """Read every 'cat' box (xmin, xmax, ymin, ymax) from one VOC-style xml file."""
    xml_path = Path(xml_path)
    image_id = xml_path.stem.split(".")[1]          # cat.12.xml -> "12"
    boxes, labels, ids = [], [], []

    for obj in ET.parse(xml_path).getroot().findall("object"):
        if obj.find("name").text != "cat":
            continue
        for bnd in obj.findall("bndbox"):
            xmin = int(float(bnd.find("xmin").text))
            xmax = int(float(bnd.find("xmax").text))
            ymin = int(float(bnd.find("ymin").text))
            ymax = int(float(bnd.find("ymax").text))
            if xmax <= xmin or ymax <= ymin:        # skip invalid boxes
                continue
            boxes.append(np.array([xmin, xmax, ymin, ymax]))
            labels.append("cat")
            ids.append(image_id)
    return np.array(boxes), np.array(labels), np.array(ids)


def region_proposals(image, max_proposals=None):
    """Selective Search (fast mode). Returns boxes as (x1, x2, y1, y2)."""
    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(image)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()                            # (x, y, w, h)
    if max_proposals is not None:
        rects = rects[:max_proposals]
    x, y, w, h = rects[:, 0], rects[:, 1], rects[:, 2], rects[:, 3]
    return np.stack([x, x + w, y, y + h], axis=1)


# Inspired by https://stackoverflow.com/questions/25349178/
def _get_iou(bb1, bb2):
    """Intersection-over-union of two boxes in (x1, x2, y1, y2) format."""
    x_left, x_right = max(bb1[0], bb2[0]), min(bb1[1], bb2[1])
    y_top, y_bottom = max(bb1[2], bb2[2]), min(bb1[3], bb2[3])
    if x_right <= x_left or y_bottom <= y_top:
        return 0.0
    intersection = (x_right - x_left) * (y_bottom - y_top)
    area1 = (bb1[1] - bb1[0]) * (bb1[3] - bb1[2])
    area2 = (bb2[1] - bb2[0]) * (bb2[3] - bb2[2])
    return intersection / float(area1 + area2 - intersection)


def save_crops(image, boxes, image_id, proposals, out_dir=TRAIN_DIR):
    """Save proposals as training crops.
    cat         : IoU > 0.5 with any ground-truth box
    no_category : IoU < 0.1 with ALL ground-truth boxes (max 50 per image, at least 20x20 px)
    `image` must be RGB.
    """
    (out_dir / "cat").mkdir(parents=True, exist_ok=True)
    (out_dir / "no_category").mkdir(parents=True, exist_ok=True)
    n_pos = n_neg = 0

    for rect in proposals:
        x1, x2, y1, y2 = (int(v) for v in rect)
        best_iou = max(_get_iou(box, (x1, x2, y1, y2)) for box in boxes)

        if best_iou > 0.5:
            n_pos += 1
            file_name = out_dir / "cat" / f"{image_id}_{n_pos}.jpg"
        elif best_iou < 0.1 and n_neg < 50 and (y2 - y1) > 20 and (x2 - x1) > 20:
            n_neg += 1
            file_name = out_dir / "no_category" / f"{image_id}_{n_neg}.jpg"
        else:
            continue

        crop = Image.fromarray(image[y1:y2, x1:x2]).resize(IMG_SIZE, Image.Resampling.LANCZOS)
        crop.save(file_name)

## 2. Look at one dataset image

In [ ]:
cat_images = sorted(DATA_DIR.rglob("cat.*.jpg"))
if not cat_images:
    raise FileNotFoundError(f"No cat.*.jpg files found under '{DATA_DIR}/'. Download the dataset first (see README).")

plt.imshow(plt.imread(cat_images[0]))
plt.title(cat_images[0].name)
plt.axis("off")
plt.show()

## 3. Create the training crops

Every Selective Search proposal is compared with the ground-truth box(es). This is slow (~5 s per image on CPU), so it is skipped if the `training/` folder already exists.

In [ ]:
def find_image_xml_pairs(data_dir):
    """Match every annotation file with the image that has the same name."""
    pairs = []
    for xml_path in sorted(data_dir.rglob("cat.*.xml")):
        jpg_path = xml_path.with_suffix(".jpg")
        if jpg_path.exists():
            pairs.append((jpg_path, xml_path))
    return pairs


pairs = find_image_xml_pairs(DATA_DIR)
if not pairs:
    raise FileNotFoundError(f"No cat.*.xml annotation files found under '{DATA_DIR}/'.")
print(f"{len(pairs)} annotated cat images found")

if REGENERATE_CROPS and TRAIN_DIR.exists():
    shutil.rmtree(TRAIN_DIR)

if TRAIN_DIR.exists() and any(TRAIN_DIR.rglob("*.jpg")):
    print(f"'{TRAIN_DIR}/' already contains crops - skipping. Set REGENERATE_CROPS = True to rebuild.")
else:
    processed = 0
    for image_path, xml_path in tqdm(pairs):
        if MAX_IMAGES is not None and processed >= MAX_IMAGES:
            break
        if image_path.stat().st_size >= MAX_FILE_SIZE:
            continue
        boxes, labels, ids = extract_ground_truth(xml_path)
        if len(boxes) == 0:
            continue
        bgr = cv2.imread(str(image_path))
        if bgr is None:
            continue
        image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)   # cv2 loads BGR, PIL/Keras expect RGB
        save_crops(image, boxes, ids[0], region_proposals(image))
        processed += 1
    print(f"Processed {processed} images")

## 4. Train / validation / test split — **by image**

Crops of the same photo look almost identical, so they must never be spread over train *and* test. The split is done on image ids (70 % / 15 % / 15 %).

In [ ]:
CLASS_NAMES = ["cat", "no_category"]      # label 0 = cat, label 1 = no_category


def list_crops(train_dir=TRAIN_DIR):
    items = []
    for label, class_name in enumerate(CLASS_NAMES):
        for path in sorted((train_dir / class_name).glob("*.jpg")):
            image_id = int(re.match(r"(\d+)_\d+\.jpg$", path.name).group(1))
            items.append((str(path), label, image_id))
    return items


items = list_crops()
image_ids = sorted({image_id for _, _, image_id in items})
random.Random(SEED).shuffle(image_ids)

n_test = max(1, int(0.15 * len(image_ids)))
n_val = max(1, int(0.15 * len(image_ids)))
split_ids = {
    "test": set(image_ids[:n_test]),
    "val": set(image_ids[n_test:n_test + n_val]),
    "train": set(image_ids[n_test + n_val:]),
}
subsets = {name: [it for it in items if it[2] in ids] for name, ids in split_ids.items()}

for name, subset in subsets.items():
    n_cat = sum(1 for _, label, _ in subset if label == 0)
    print(f"{name:5s}: {len(split_ids[name]):4d} images | {len(subset):6d} crops "
          f"({n_cat} cat, {len(subset) - n_cat} no_category)")


def load_crop(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(image, IMG_SIZE), label


def make_dataset(subset, training):
    paths = [p for p, _, _ in subset]
    labels = [l for _, l, _ in subset]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
    return ds.map(load_crop, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(subsets["train"], training=True)
val_ds = make_dataset(subsets["val"], training=False)
test_ds = make_dataset(subsets["test"], training=False)

## 5. Model

MobileNetV2 (ImageNet weights, frozen) + small classification head. Augmentation (random crop + horizontal flip) is only active during training. `Rescaling(1/127.5, offset=-1)` maps pixels to [-1, 1], which is what MobileNetV2 expects.

In [ ]:
# Inspired by https://pyimagesearch.com/2020/07/13/r-cnn-object-detection-with-keras-tensorflow-and-deep-learning/
inputs = keras.layers.Input(shape=(*IMG_SIZE, 3))
x = keras.layers.ZeroPadding2D((16, 16))(inputs)
x = keras.layers.RandomCrop(*IMG_SIZE)(x)
x = keras.layers.RandomFlip("horizontal")(x)
x = keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)

base_model = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_tensor=x)
base_model.trainable = False

x = keras.layers.AveragePooling2D(pool_size=(7, 7))(base_model.output)
x = keras.layers.Flatten(name="flatten")(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(2, activation="softmax")(x)

model = keras.models.Model(inputs=inputs, outputs=outputs)
model.summary()

## 6. Train

In [ ]:
# Background crops outnumber cat crops, so cat errors are weighted 5x.
class_weight = {0: 5, 1: 1}

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

model.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(history.history[metric], label=f"train {metric}")
    ax.plot(history.history[f"val_{metric}"], label=f"val {metric}")
    ax.set_xlabel("Epoch")
    ax.set_title(metric.capitalize())
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate on the held-out test images

Accuracy alone is misleading here (most crops are background), so look at precision / recall of the **cat** class.

In [ ]:
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(probs.argmax(axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print("Confusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(y_true, y_pred))

## 8. Detect cats in a new image

Runs Selective Search on `sample_input.jpg`, classifies every proposal, and keeps the best boxes with Non-Maximum Suppression. Only boxes with a cat probability above the threshold are drawn, so an image without a cat should give no boxes.

In [ ]:
# Not needed if the model was just trained in this session
if "model" not in globals():
    model = keras.models.load_model(MODEL_PATH)

if not SAMPLE_IMAGE.exists():
    raise FileNotFoundError(f"Put a photo named '{SAMPLE_IMAGE}' next to the notebook (use your own or a copyright-free image).")

image = np.array(Image.open(SAMPLE_IMAGE).convert("RGB"))

# Shrink large photos: far fewer proposals, much faster, same result
MAX_SIDE = 640
scale = MAX_SIDE / max(image.shape[:2])
if scale < 1:
    image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

plt.imshow(image)
plt.axis("off")
plt.title("Input image")
plt.show()

In [ ]:
def detect_cats(image, model, score_threshold=0.9, iou_threshold=0.3, max_boxes=5, min_size=20):
    """Return a list of ((x1, x2, y1, y2), cat_probability) after NMS."""
    proposals = region_proposals(image, max_proposals=2000)
    widths = proposals[:, 1] - proposals[:, 0]
    heights = proposals[:, 3] - proposals[:, 2]
    proposals = proposals[(widths >= min_size) & (heights >= min_size)]
    if len(proposals) == 0:
        return []

    crops = np.stack([
        np.array(Image.fromarray(image[y1:y2, x1:x2]).resize(IMG_SIZE, Image.Resampling.LANCZOS))
        for x1, x2, y1, y2 in proposals
    ])
    scores = model.predict(crops, batch_size=64, verbose=0)[:, 0]      # class 0 = cat

    # tf NMS expects boxes as (y1, x1, y2, x2)
    boxes_yx = np.stack([proposals[:, 2], proposals[:, 0], proposals[:, 3], proposals[:, 1]], axis=1).astype("float32")
    keep = tf.image.non_max_suppression(
        boxes_yx, scores.astype("float32"),
        max_output_size=max_boxes, iou_threshold=iou_threshold, score_threshold=score_threshold,
    ).numpy()
    return [(proposals[i], float(scores[i])) for i in keep]


detections = detect_cats(image, model)

result = image.copy()
for (x1, x2, y1, y2), score in detections:
    cv2.rectangle(result, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 3)
    cv2.putText(result, f"cat {score:.2f}", (int(x1), max(int(y1) - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    print(f"cat  p={score:.3f}  box=(x1={x1}, y1={y1}, x2={x2}, y2={y2})")
if not detections:
    print("No cat detected")

plt.imshow(result)
plt.axis("off")
plt.title(f"{len(detections)} detection(s)")
plt.show()

Path("results").mkdir(exist_ok=True)
Image.fromarray(result).save("results/detection_result.jpg")